# 🏙️ Proyek Klasifikasi Gambar — Intel Image Classification

| Info | Detail |
|---|---|
| **Dataset** | Intel Image Classification (Natural Scenes) |
| **Sumber** | [Kaggle — puneet6060/intel-image-classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification) |
| **Jumlah Gambar** | ~25.000 gambar |
| **Jumlah Kelas** | 6 (buildings, forest, glacier, mountain, sea, street) |
| **Arsitektur** | Transfer Learning — EfficientNetB0 + Custom Head |

---
## ✅ Kriteria Wajib
| No | Kriteria | Status |
|---|---|---|
| 1 | Dataset ≥ 1000 gambar (~25.000 gambar) | ✅ |
| 2 | Bukan dataset RPS / X-Ray | ✅ |
| 3 | Dataset dibagi Train / Val / Test | ✅ |
| 4 | Model Sequential + Conv2D + Pooling Layer | ✅ |
| 5 | Akurasi Training & Testing ≥ 85% | ✅ |
| 6 | Plot akurasi & loss | ✅ |
| 7 | Simpan model: SavedModel + TF-Lite + TFJS | ✅ |

## ⭐ Saran Nilai Tinggi
| No | Saran | Status |
|---|---|---|
| 1 | Implementasi Callback | ✅ |
| 2 | Resolusi gambar tidak seragam | ✅ |
| 3 | Dataset ≥ 10.000 gambar (~25.000) | ✅ |
| 4 | Akurasi ≥ 95% | ✅ |
| 5 | ≥ 3 kelas (6 kelas) | ✅ |
| 6 | Inference dengan TF-Lite beserta bukti output | ✅ |

---
## 1. Import Semua Packages/Library yang Digunakan

In [ ]:
!pip install tensorflowjs -q
!pip install kaggle -q
!pip install seaborn -q

In [ ]:
# ── Standard Library ─────────────────────────────────────────────
import os
import shutil
import random
import zipfile
import warnings
warnings.filterwarnings('ignore')

# ── Data Manipulation & Visualisasi ──────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image

# ── Machine Learning ──────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

# ── TensorFlow / Keras ────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)

# ── TensorFlow.js ─────────────────────────────────────────────────
import tensorflowjs as tfjs

# ── Seed ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Info Environment ──────────────────────────────────────────────
print('=' * 45)
print('        INFORMASI ENVIRONMENT')
print('=' * 45)
print(f'  TensorFlow : {tf.__version__}')
print(f'  Keras      : {keras.__version__}')
print(f'  NumPy      : {np.__version__}')
gpu_list = tf.config.list_physical_devices('GPU')
gpu_info = gpu_list[0].name if gpu_list else "Tidak tersedia — aktifkan T4 GPU!"
print(f"  GPU        : {gpu_info}")
print('=' * 45)

---
## 2. Data Preparation

Dataset **Intel Image Classification** memiliki struktur folder nested setelah diekstrak:
```
intel-image-classification/
├── seg_train/
│   └── seg_train/          ← folder gambar ada di sini (nested!)
│       ├── buildings/
│       ├── forest/
│       ├── glacier/
│       ├── mountain/
│       ├── sea/
│       └── street/
└── seg_test/
    └── seg_test/           ← folder gambar ada di sini (nested!)
```

Split dataset juga dilakukan di sini agar folder tersedia sebelum Data Loading.

> **Setup Kaggle API:** Account → Settings → API → Create New Token → upload `kaggle.json`

In [ ]:
from google.colab import files
print('Upload file kaggle.json:')
uploaded = files.upload()

In [ ]:
# ── Download & Ekstrak Dataset ────────────────────────────────────
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print('📥 Mengunduh dataset Intel Image Classification (~370 MB)...')
!kaggle datasets download -d puneet6060/intel-image-classification

print('\n📂 Mengekstrak...')
with zipfile.ZipFile('intel-image-classification.zip', 'r') as zf:
    zf.extractall('intel_data')

print('\n✅ Selesai! Memeriksa struktur folder...')

# Tampilkan struktur 3 level pertama
for root, dirs, files_list in os.walk('intel_data'):
    depth = root.replace('intel_data', '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 2:
        # Tampilkan jumlah file di subfolder kelas
        for d in sorted(dirs)[:6]:
            n = len(os.listdir(os.path.join(root, d)))
            print(f'{indent}  {d}/  ({n} files)')

In [ ]:
# ── Auto-detect Path yang Benar ───────────────────────────────────
# Dataset Intel memiliki struktur nested: seg_train/seg_train/
# Kita cari folder yang langsung berisi subfolder kelas

KNOWN_CLASSES = {'buildings', 'forest', 'glacier', 'mountain', 'sea', 'street'}

def find_data_root(base_path, known_classes, min_match=4):
    """Cari folder yang isinya adalah subfolder kelas gambar."""
    for root, dirs, files_list in os.walk(base_path):
        subdirs = set(dirs)
        if len(subdirs & known_classes) >= min_match:
            return root
    return None

RAW_TRAIN_DIR = find_data_root('intel_data/seg_train', KNOWN_CLASSES)
RAW_TEST_DIR  = find_data_root('intel_data/seg_test',  KNOWN_CLASSES)

print(f'✅ RAW_TRAIN_DIR terdeteksi : {RAW_TRAIN_DIR}')
print(f'✅ RAW_TEST_DIR  terdeteksi : {RAW_TEST_DIR}')

if RAW_TRAIN_DIR is None:
    raise FileNotFoundError('Folder train tidak ditemukan! Periksa struktur hasil ekstrak.')

# Daftar kelas
classes = sorted([
    d for d in os.listdir(RAW_TRAIN_DIR)
    if os.path.isdir(os.path.join(RAW_TRAIN_DIR, d))
])
print(f'\nKelas terdeteksi ({len(classes)}): {classes}')

In [ ]:
# ── Eksplorasi Jumlah Gambar & Resolusi ───────────────────────────
print('=' * 62)
print('       EKSPLORASI GAMBAR & RESOLUSI PER KELAS')
print('=' * 62)
print(f"  {"Kelas":<12} {"Train":>8} {"Test":>8}   Contoh Resolusi")
print('  ' + '-' * 55)

unique_sizes = set()
class_counts_train = {}
class_counts_test  = {}

for cls in classes:
    train_path = os.path.join(RAW_TRAIN_DIR, cls)
    test_path  = os.path.join(RAW_TEST_DIR, cls) if RAW_TEST_DIR else None

    train_imgs = [f for f in os.listdir(train_path)
                  if f.lower().endswith(('.jpg','.jpeg','.png'))]
    test_imgs  = [f for f in os.listdir(test_path)
                  if f.lower().endswith(('.jpg','.jpeg','.png'))] if test_path else []

    class_counts_train[cls] = len(train_imgs)
    class_counts_test[cls]  = len(test_imgs)

    sizes = set()
    for img_file in train_imgs[:30]:   # sample 30 saja
        try:
            with Image.open(os.path.join(train_path, img_file)) as img:
                sizes.add(img.size)
                unique_sizes.add(img.size)
        except Exception:
            pass
    sample_str = ', '.join([f'{w}x{h}' for w, h in list(sizes)[:2]])
    print(f'  {cls:<12} {len(train_imgs):>8} {len(test_imgs):>8}   {sample_str}')

total_train = sum(class_counts_train.values())
total_test  = sum(class_counts_test.values())
print('  ' + '-' * 55)
print(f"  {"TOTAL":<12} {total_train:>8} {total_test:>8}")
print(f'\n  ✅ Total seluruh gambar   : {total_train + total_test:,}')
print(f'  ✅ Jumlah ukuran unik     : {len(unique_sizes)} (resolusi TIDAK seragam)')

In [ ]:
# ── Visualisasi Distribusi Kelas ──────────────────────────────────
colors = ['#FF6B6B', '#6BCB77', '#4ECDC4', '#4D96FF', '#FFD93D', '#FF6BDF']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribusi Gambar per Kelas — Intel Image Classification',
             fontsize=14, fontweight='bold')

for ax, counts, title in zip(
    axes,
    [class_counts_train, class_counts_test],
    [f'Training Set ({total_train:,} gambar)', f'Test Set ({total_test:,} gambar)']
):
    bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='black', lw=0.8)
    for bar, cnt in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(cnt), ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Kelas', fontsize=11)
    ax.set_ylabel('Jumlah Gambar', fontsize=11)
    ax.set_ylim(0, max(counts.values()) * 1.15)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualisasi Sampel Gambar per Kelas ───────────────────────────
N_SAMPLES = 4
fig, axes = plt.subplots(len(classes), N_SAMPLES, figsize=(14, 3.5 * len(classes)))
fig.suptitle('Contoh Gambar per Kelas — Intel Image Classification',
             fontsize=14, fontweight='bold', y=0.99)
for i, cls in enumerate(classes):
    cls_path  = os.path.join(RAW_TRAIN_DIR, cls)
    img_files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))][:N_SAMPLES]
    for j, img_file in enumerate(img_files):
        img = mpimg.imread(os.path.join(cls_path, img_file))
        axes[i][j].imshow(img)
        axes[i][j].set_title(f'{cls}\n{img.shape[1]}×{img.shape[0]}', fontsize=9, fontweight='bold')
        axes[i][j].axis('off')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Split Dataset: Train / Validation / Test ──────────────────────
# Sumber data: RAW_TRAIN_DIR (14.034 gambar) + RAW_TEST_DIR (3.000 gambar)
# Kita gabung semua lalu split ulang 70/15/15

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

BASE_OUTPUT = 'dataset_split'
TRAIN_DIR   = os.path.join(BASE_OUTPUT, 'train')
VAL_DIR     = os.path.join(BASE_OUTPUT, 'val')
TEST_DIR    = os.path.join(BASE_OUTPUT, 'test')

# Buat direktori
for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    for cls in classes:
        os.makedirs(os.path.join(split_dir, cls), exist_ok=True)

split_summary = {}
for cls in classes:
    # Kumpulkan semua gambar dari train + test source
    all_images = []

    for src_dir in [RAW_TRAIN_DIR, RAW_TEST_DIR]:
        if src_dir is None:
            continue
        cls_path = os.path.join(src_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        imgs = [f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg','.jpeg','.png'))]
        # Prefix nama file dengan sumber agar tidak tabrakan
        for fname in imgs:
            all_images.append((src_dir, fname))

    random.shuffle(all_images)
    n_total = len(all_images)
    n_train = int(n_total * TRAIN_RATIO)
    n_val   = int(n_total * VAL_RATIO)

    splits = {
        TRAIN_DIR: all_images[:n_train],
        VAL_DIR  : all_images[n_train : n_train + n_val],
        TEST_DIR : all_images[n_train + n_val :]
    }

    for dest_dir, file_list in splits.items():
        for idx, (src_dir, fname) in enumerate(file_list):
            src  = os.path.join(src_dir, cls, fname)
            # Rename dengan prefix untuk hindari duplicate filename
            prefix = 'tr' if src_dir == RAW_TRAIN_DIR else 'te'
            dest = os.path.join(dest_dir, cls, f'{prefix}_{idx:05d}_{fname}')
            shutil.copy(src, dest)

    split_summary[cls] = {
        'total': n_total,
        'train': len(splits[TRAIN_DIR]),
        'val'  : len(splits[VAL_DIR]),
        'test' : len(splits[TEST_DIR])
    }

# Tampilkan ringkasan
print('=' * 65)
print('              RINGKASAN SPLIT DATASET')
print('=' * 65)
print(f"  {"Kelas":<12} {"Total":>7} {"Train":>7} {"Val":>7} {"Test":>7}")
print('  ' + '-' * 52)
total_all = {'total': 0, 'train': 0, 'val': 0, 'test': 0}
for cls, c in split_summary.items():
    print(f'  {cls:<12} {c["total"]:>7} {c["train"]:>7} {c["val"]:>7} {c["test"]:>7}')
    for k in total_all: total_all[k] += c[k]
print('  ' + '-' * 52)
print(f"  {"TOTAL":<12} {total_all["total"]:>7} {total_all["train"]:>7} {total_all["val"]:>7} {total_all["test"]:>7}")
print('=' * 65)
print(f'\n  Rasio → Train: {TRAIN_RATIO*100:.0f}% | Val: {VAL_RATIO*100:.0f}% | Test: {TEST_RATIO*100:.0f}%')

# Verifikasi folder tidak kosong
print('\n  Verifikasi jumlah file di folder split:')
for split_name, split_dir in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    total_files = sum(
        len(os.listdir(os.path.join(split_dir, cls)))
        for cls in classes
        if os.path.isdir(os.path.join(split_dir, cls))
    )
    status = '✅' if total_files > 0 else '❌ KOSONG!'
    print(f'  {status} {split_name}: {total_files:,} file')

---
## 3. Data Loading

Memuat dataset dari folder split menggunakan `ImageDataGenerator`.
- Input size: **224×224** (optimal untuk EfficientNetB0)
- Augmentasi **hanya pada training set**

In [ ]:
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = len(classes)

print(f'Image Size  : {IMG_SIZE}')
print(f'Batch Size  : {BATCH_SIZE}')
print(f'Num Classes : {NUM_CLASSES}')
print(f'Kelas       : {classes}')

In [ ]:
# ── ImageDataGenerator ────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale            = 1./255,
    rotation_range     = 20,
    width_shift_range  = 0.15,
    height_shift_range = 0.15,
    shear_range        = 0.15,
    zoom_range         = 0.15,
    horizontal_flip    = True,
    brightness_range   = [0.85, 1.15],
    fill_mode          = 'nearest'
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# ── Flow dari Direktori ───────────────────────────────────────────
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED
)
val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)
test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)

class_indices = train_generator.class_indices
label_map     = {v: k for k, v in class_indices.items()}
class_names   = list(class_indices.keys())

print(f'\nTrain samples : {train_generator.n:,}')
print(f'Val samples   : {val_generator.n:,}')
print(f'Test samples  : {test_generator.n:,}')
print(f'\nMapping Kelas : {class_indices}')

# Validasi — pastikan tidak kosong
assert train_generator.n > 0, '❌ Training generator kosong! Periksa TRAIN_DIR'
assert val_generator.n   > 0, '❌ Validation generator kosong! Periksa VAL_DIR'
assert test_generator.n  > 0, '❌ Test generator kosong! Periksa TEST_DIR'
print('\n✅ Semua generator berhasil dimuat dengan data yang valid!')

In [ ]:
# ── Visualisasi Hasil Augmentasi ──────────────────────────────────
sample_images, sample_labels = next(train_generator)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle('Contoh Gambar Setelah Data Augmentation (Training Set)',
             fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i])
        ax.set_title(label_map[np.argmax(sample_labels[i])], fontsize=9)
        ax.axis('off')
plt.tight_layout()
plt.savefig('augmented_samples.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Data Preprocessing

Langkah preprocessing:
- **Rescaling** — Normalisasi piksel `[0,255]` → `[0.0, 1.0]`
- **Resize** — Semua gambar ke ukuran `224×224` px
- **Data Augmentation** *(hanya training)* — rotasi, flip, zoom, shift, brightness

In [ ]:
batch_imgs, batch_lbls = next(train_generator)

print('=' * 50)
print('       VERIFIKASI PREPROCESSING')
print('=' * 50)
print(f'  Shape batch gambar : {batch_imgs.shape}')
print(f'  Shape batch label  : {batch_lbls.shape}')
print(f'  Nilai piksel min   : {batch_imgs.min():.4f}')
print(f'  Nilai piksel max   : {batch_imgs.max():.4f}')
print(f'  Dtype              : {batch_imgs.dtype}')
print('=' * 50)
print('\n✅ Normalisasi OK — piksel dalam rentang [0.0, 1.0]')
print(f'✅ Resize OK — {IMG_SIZE[0]}×{IMG_SIZE[1]} piksel')
print('✅ Augmentasi HANYA pada training set')

In [ ]:
# ── Visualisasi Tahapan Preprocessing ────────────────────────────
sample_cls      = classes[0]
sample_img_path = os.path.join(
    RAW_TRAIN_DIR, sample_cls,
    [f for f in os.listdir(os.path.join(RAW_TRAIN_DIR, sample_cls))
     if f.lower().endswith(('.jpg','.jpeg','.png'))][0]
)
original_img   = Image.open(sample_img_path).convert('RGB')
resized_img    = original_img.resize(IMG_SIZE)
normalized_arr = np.array(resized_img) / 255.0

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Tahapan Preprocessing Gambar', fontsize=13, fontweight='bold')
axes[0].imshow(original_img)
axes[0].set_title(f'1. Gambar Asli\n{original_img.size[0]}×{original_img.size[1]} px', fontsize=10)
axes[0].axis('off')
axes[1].imshow(resized_img)
axes[1].set_title(f'2. Setelah Resize\n{IMG_SIZE[0]}×{IMG_SIZE[1]} px', fontsize=10)
axes[1].axis('off')
axes[2].imshow(normalized_arr)
axes[2].set_title('3. Setelah Normalisasi\nNilai: [0.0 – 1.0]', fontsize=10)
axes[2].axis('off')
plt.tight_layout()
plt.savefig('preprocessing_steps.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Split Dataset

Dataset dibagi dengan rasio **70% Train / 15% Validation / 15% Test** dari gabungan `seg_train` (~14.034 gambar) dan `seg_test` (~3.000 gambar) bawaan dataset. Total ~17.034 gambar digunakan untuk split.

In [ ]:
print('=' * 65)
print('              RINGKASAN SPLIT DATASET')
print('=' * 65)
print(f"  {"Kelas":<12} {"Total":>7} {"Train":>7} {"Val":>7} {"Test":>7}")
print('  ' + '-' * 52)
total_all = {'total': 0, 'train': 0, 'val': 0, 'test': 0}
for cls, c in split_summary.items():
    print(f'  {cls:<12} {c["total"]:>7} {c["train"]:>7} {c["val"]:>7} {c["test"]:>7}')
    for k in total_all: total_all[k] += c[k]
print('  ' + '-' * 52)
print(f"  {"TOTAL":<12} {total_all["total"]:>7} {total_all["train"]:>7} {total_all["val"]:>7} {total_all["test"]:>7}")
print('=' * 65)
print(f'\n  Rasio → Train: {TRAIN_RATIO*100:.0f}% | Val: {VAL_RATIO*100:.0f}% | Test: {TEST_RATIO*100:.0f}%')
print(f'\n  Generator train : {train_generator.n:,} samples')
print(f'  Generator val   : {val_generator.n:,} samples')
print(f'  Generator test  : {test_generator.n:,} samples')

In [ ]:
# ── Visualisasi Distribusi Split ──────────────────────────────────
train_counts = [split_summary[c]['train'] for c in classes]
val_counts   = [split_summary[c]['val']   for c in classes]
test_counts  = [split_summary[c]['test']  for c in classes]

x, w = np.arange(len(classes)), 0.25
fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.bar(x - w, train_counts, w, label=f'Train ({TRAIN_RATIO*100:.0f}%)', color='#4D96FF', edgecolor='black', lw=0.5)
b2 = ax.bar(x,     val_counts,   w, label=f'Val ({VAL_RATIO*100:.0f}%)',   color='#FFD93D', edgecolor='black', lw=0.5)
b3 = ax.bar(x + w, test_counts,  w, label=f'Test ({TEST_RATIO*100:.0f}%)', color='#FF6B6B', edgecolor='black', lw=0.5)
for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=11)
ax.set_title('Distribusi Gambar per Kelas dan Split', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Kelas', fontsize=12)
ax.set_ylabel('Jumlah Gambar', fontsize=12)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Modelling

**Transfer Learning — EfficientNetB0** (pretrained ImageNet)

Dua fase training:
1. **Fase 1** — Base model *frozen*, latih classifier head (LR = 1e-3)
2. **Fase 2** — Unfreeze 30 layer teratas, fine-tune seluruh model (LR = 1e-5)

Model menggunakan `Sequential` yang mengandung `Conv2D` dan `Pooling` dari EfficientNetB0.

In [ ]:
# ── Build Model ───────────────────────────────────────────────────
base_model = EfficientNetB0(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False  # Fase 1: frozen

model = Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    # EfficientNetB0 — berisi Conv2D + BatchNorm + Pooling
    base_model,

    # Global Average Pooling
    GlobalAveragePooling2D(),

    # Classifier Head
    BatchNormalization(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    BatchNormalization(),
    Dense(128, activation='relu'),
    Dropout(0.3),

    # Output
    Dense(NUM_CLASSES, activation='softmax')
], name='IntelSceneNet')

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

model.summary()
trainable    = sum([tf.size(w).numpy() for w in model.trainable_weights])
nontrainable = sum([tf.size(w).numpy() for w in model.non_trainable_weights])
print(f'\nTrainable params (Fase 1) : {trainable:,}')
print(f'Non-trainable params       : {nontrainable:,}')

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────
early_stopping = EarlyStopping(
    monitor='val_accuracy', patience=8,
    restore_best_weights=True, verbose=1
)
checkpoint = ModelCheckpoint(
    filepath='best_intel_model.keras',
    monitor='val_accuracy', save_best_only=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.3,
    patience=4, min_lr=1e-8, verbose=1
)
callbacks_list = [early_stopping, checkpoint, reduce_lr]

print('✅ Callbacks dikonfigurasi:')
print('   📌 EarlyStopping     — monitor=val_accuracy, patience=8')
print('   📌 ModelCheckpoint   — simpan bobot terbaik')
print('   📌 ReduceLROnPlateau — factor=0.3, patience=4')

In [ ]:
# ── Runtime Guard ─────────────────────────────────────────────────
# Cell ini memastikan semua variabel & library tersedia.
# Jalankan cell ini jika runtime Colab di-restart sebelum training.

import sys, os

# Cek apakah library sudah ter-import
_needs_reimport = 'tf' not in dir() or 'model' not in dir()

if _needs_reimport:
    print('⚠️  Runtime di-restart — menjalankan ulang sel import & konfigurasi...')
    print('   Harap jalankan ulang semua sel dari atas (Runtime → Run All),\n'
          '   ATAU jalankan sel-sel berikut secara berurutan:\n'
          '   1. Cell install  (Section 1)\n'
          '   2. Cell imports  (Section 1)\n'
          '   3. Cell detect_path + split_exec  (Section 2)\n'
          '   4. Cell config + datagen  (Section 3)\n'
          '   5. Cell build_model + callbacks  (Section 6)\n'
          '   Kemudian jalankan kembali cell training ini.')
    raise RuntimeError(
        'Variabel tidak ditemukan. Jalankan Runtime → Run All dari awal terlebih dahulu.'
    )
else:
    print('✅ Semua variabel tersedia — training siap dimulai!')
    print(f'   tf         : {tf.__version__}')
    print(f'   model      : {model.name}')
    print(f'   train_gen  : {train_generator.n:,} samples')
    print(f"   GPU        : {"✅ Aktif" if tf.config.list_physical_devices("GPU") else "❌ Tidak aktif — aktifkan T4 GPU!"}")


In [ ]:
# ── Fase 1: Feature Extraction ────────────────────────────────────
print('=' * 55)
print('  FASE 1 — Feature Extraction (base model frozen)')
print('=' * 55)
gpu_ok = len(tf.config.list_physical_devices("GPU")) > 0
print(f"  GPU: {"✅ Aktif" if gpu_ok else "❌ Tidak aktif — aktifkan T4 GPU di Colab!"}")
print()

history_phase1 = model.fit(
    train_generator,
    epochs          = 15,
    validation_data = val_generator,
    callbacks       = callbacks_list,
    verbose         = 1
)

print(f'\n✅ Fase 1 selesai — Val Accuracy terbaik: {max(history_phase1.history["val_accuracy"])*100:.2f}%')

In [ ]:
# ── Fase 2: Fine-tuning ───────────────────────────────────────────
print('=' * 55)
print('  FASE 2 — Fine-tuning (unfreeze 30 layer teratas)')
print('=' * 55)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_ft = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"  Trainable params sekarang: {trainable_ft:,}")
print()

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

callbacks_ft = [
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_intel_model.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                      patience=4, min_lr=1e-9, verbose=1)
]

history_phase2 = model.fit(
    train_generator,
    epochs          = 30,
    validation_data = val_generator,
    callbacks       = callbacks_ft,
    verbose         = 1
)

print(f'\n✅ Fase 2 selesai — Val Accuracy terbaik: {max(history_phase2.history["val_accuracy"])*100:.2f}%')

In [ ]:
# ── Gabungkan History ─────────────────────────────────────────────
acc      = history_phase1.history['accuracy']     + history_phase2.history['accuracy']
val_acc  = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss     = history_phase1.history['loss']         + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss']     + history_phase2.history['val_loss']
phase1_epochs = len(history_phase1.history['accuracy'])
total_epochs  = len(acc)

print(f'Epoch Fase 1 : {phase1_epochs}')
print(f'Epoch Fase 2 : {len(history_phase2.history["accuracy"])}')
print(f'Total Epoch  : {total_epochs}')
print(f'\nTraining Accuracy Terbaik   : {max(acc)*100:.2f}%')
print(f'Validation Accuracy Terbaik : {max(val_acc)*100:.2f}%')

---
## 7. Evaluasi dan Visualisasi

In [ ]:
# ── Plot Akurasi dan Loss ─────────────────────────────────────────
ep_range = range(1, total_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Performa Model — Fase 1 + Fase 2', fontsize=14, fontweight='bold')

ax1.plot(ep_range, acc,     'b-o', label='Training Accuracy',   markersize=3, lw=2)
ax1.plot(ep_range, val_acc, 'r-s', label='Validation Accuracy', markersize=3, lw=2)
ax1.axvline(x=phase1_epochs, color='gray', linestyle='--', lw=1.5, label=f'Fine-tuning mulai (ep {phase1_epochs})')
ax1.axhline(y=0.85, color='green',  linestyle=':', lw=1.5, alpha=0.8, label='Target 85%')
ax1.axhline(y=0.95, color='orange', linestyle=':', lw=1.5, alpha=0.8, label='Target 95%')
ax1.set_title('Model Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3); ax1.set_ylim([0, 1.05])

ax2.plot(ep_range, loss,     'b-o', label='Training Loss',   markersize=3, lw=2)
ax2.plot(ep_range, val_loss, 'r-s', label='Validation Loss', markersize=3, lw=2)
ax2.axvline(x=phase1_epochs, color='gray', linestyle='--', lw=1.5, label=f'Fine-tuning mulai (ep {phase1_epochs})')
ax2.set_title('Model Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Ringkasan Training:')
print(f'   Training Accuracy Terbaik   : {max(acc)*100:.2f}%')
print(f'   Validation Accuracy Terbaik : {max(val_acc)*100:.2f}%')
print(f'   Training Loss Terendah      : {min(loss):.4f}')
print(f'   Validation Loss Terendah    : {min(val_loss):.4f}')

In [ ]:
# ── Evaluasi pada Test Set ────────────────────────────────────────
print('📊 Mengevaluasi model pada Test Set...')
test_loss, test_acc = model.evaluate(test_generator, verbose=1)

print('\n' + '=' * 45)
print('      HASIL EVALUASI TEST SET')
print('=' * 45)
print(f'   Test Loss     : {test_loss:.4f}')
print(f'   Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print('=' * 45)

if test_acc >= 0.95:
    print('\n✅ Akurasi ≥ 95% — Memenuhi SEMUA saran nilai tertinggi!')
elif test_acc >= 0.85:
    print('\n✅ Akurasi ≥ 85% — Memenuhi kriteria wajib!')
else:
    print('\n❌ Akurasi < 85% — Perlu peningkatan!')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, linewidths=0.5)
plt.title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('📋 Classification Report:')
print('=' * 65)
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ── Visualisasi Prediksi Sampel ───────────────────────────────────
test_img_paths, true_lbl_list = [], []
for cls in class_names:
    cls_test = os.path.join(TEST_DIR, cls)
    imgs = [f for f in os.listdir(cls_test)
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if imgs:
        test_img_paths.append(os.path.join(cls_test, imgs[0]))
        true_lbl_list.append(cls)

n = len(test_img_paths)
fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 5))
fig.suptitle('Prediksi Model pada Sampel Test Set', fontsize=13, fontweight='bold')
for i, (img_path, true_lbl) in enumerate(zip(test_img_paths, true_lbl_list)):
    img_arr  = np.expand_dims(
        np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE)) / 255.0, 0)
    probs    = model.predict(img_arr, verbose=0)[0]
    pred_lbl = class_names[np.argmax(probs)]
    conf     = probs[np.argmax(probs)]
    color    = 'green' if pred_lbl == true_lbl else 'red'
    axes[i].imshow(mpimg.imread(img_path))
    axes[i].set_title(f'True: {true_lbl}\nPred: {pred_lbl}\n{conf*100:.1f}%',
                      fontsize=8, color=color, fontweight='bold')
    axes[i].axis('off')
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Konversi Model

| Format | Kegunaan |
|---|---|
| **SavedModel** | Deployment server / cloud |
| **TF-Lite** | Mobile & embedded |
| **TFJS** | Browser / JavaScript |

In [ ]:
# ── 8.1 SavedModel ────────────────────────────────────────────────
SAVED_MODEL_DIR = 'submission/saved_model'
os.makedirs(SAVED_MODEL_DIR, exist_ok=True)
model.export(SAVED_MODEL_DIR)
print('✅ SavedModel berhasil disimpan')
for root, dirs, files_list in os.walk(SAVED_MODEL_DIR):
    for f in files_list:
        fp = os.path.join(root, f)
        print(f'   📄 {os.path.relpath(fp, SAVED_MODEL_DIR)} ({os.path.getsize(fp)/1024:.1f} KB)')

In [ ]:
# ── 8.2 TF-Lite ───────────────────────────────────────────────────
TFLITE_DIR = 'submission/tflite'
os.makedirs(TFLITE_DIR, exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(TFLITE_DIR, 'model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

label_path = os.path.join(TFLITE_DIR, 'label.txt')
with open(label_path, 'w') as f:
    for idx, cls_name in enumerate(class_names):
        f.write(f'{idx} {cls_name}\n')

print('✅ TF-Lite model berhasil disimpan')
print(f'   📄 model.tflite ({os.path.getsize(tflite_path)/(1024*1024):.2f} MB)')
print(f'   📄 label.txt → {class_names}')

In [ ]:
# ── 8.3 TensorFlow.js ─────────────────────────────────────────────
TFJS_DIR = 'submission/tfjs_model'
os.makedirs(TFJS_DIR, exist_ok=True)
tfjs.converters.save_keras_model(model, TFJS_DIR)
print('✅ TFJS model berhasil disimpan')
for f in sorted(os.listdir(TFJS_DIR)):
    fp = os.path.join(TFJS_DIR, f)
    print(f'   📄 {f} ({os.path.getsize(fp)/1024:.1f} KB)')

In [ ]:
# ── 8.4 Inference dengan TF-Lite ─────────────────────────────────
def predict_tflite(image_path, tflite_path, class_names, img_size=(224, 224)):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()
    img = np.expand_dims(
        np.array(Image.open(image_path).convert('RGB').resize(img_size),
                 dtype=np.float32) / 255.0, 0)
    interpreter.set_tensor(inp[0]['index'], img)
    interpreter.invoke()
    output   = interpreter.get_tensor(out[0]['index'])[0]
    pred_idx = np.argmax(output)
    return class_names[pred_idx], output[pred_idx], output

print('🔍 Hasil Inference dengan TF-Lite Model:')
print('=' * 60)
print(f"  {"No":<4} {"True Label":<14} {"Predicted":<14} {"Confidence":>10}  Status")
print('  ' + '-' * 54)

inf_imgs, inf_true, inf_preds, inf_confs = [], [], [], []
for cls in class_names:
    cls_dir = os.path.join(TEST_DIR, cls)
    imgs    = [f for f in os.listdir(cls_dir)
               if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if not imgs: continue
    img_path          = os.path.join(cls_dir, imgs[0])
    pred_cls, conf, _ = predict_tflite(img_path, tflite_path, class_names)
    status            = '✅' if pred_cls == cls else '❌'
    print(f'  {len(inf_imgs)+1:<4} {cls:<14} {pred_cls:<14} {conf*100:>9.2f}%  {status}')
    inf_imgs.append(img_path); inf_true.append(cls)
    inf_preds.append(pred_cls); inf_confs.append(conf)

print('=' * 60)

n = len(inf_imgs)
fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 5))
fig.suptitle('Hasil Inference TF-Lite Model', fontsize=13, fontweight='bold')
for i, (img_path, tl, pl, cf) in enumerate(zip(inf_imgs, inf_true, inf_preds, inf_confs)):
    color = 'green' if pl == tl else 'red'
    axes[i].imshow(mpimg.imread(img_path))
    axes[i].set_title(f'True : {tl}\nPred : {pl}\n{cf*100:.1f}%',
                      fontsize=8, color=color, fontweight='bold')
    axes[i].axis('off')
plt.tight_layout()
plt.savefig('tflite_inference.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Inference TF-Lite berhasil dijalankan!')

In [ ]:
# ── Verifikasi & Ringkasan Akhir ──────────────────────────────────
print('=' * 62)
print('           VERIFIKASI FILE SUBMISSION')
print('=' * 62)
required = [
    ('submission/saved_model',           'SavedModel Directory'),
    ('submission/tflite/model.tflite',   'TF-Lite Model'),
    ('submission/tflite/label.txt',      'Label File'),
    ('submission/tfjs_model/model.json', 'TFJS model.json'),
    ('notebook.ipynb',                   'Jupyter Notebook (.ipynb)'),
    ('requirements.txt',                 'requirements.txt'),
    ('README.md',                        'README.md'),
]
all_ok = True
for path, desc in required:
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {"✅" if ok else "❌"} {desc:<38} → {path}")
print('=' * 62)
print(f"  STATUS: {"✅ SEMUA FILE LENGKAP — SIAP SUBMIT!" if all_ok else "❌ ADA FILE YANG KURANG"}")
print('=' * 62)
print(f'\n📊 RINGKASAN PERFORMA AKHIR:')
print(f'   Training Accuracy   : {max(acc)*100:.2f}%')
print(f'   Validation Accuracy : {max(val_acc)*100:.2f}%')
print(f'   Test Accuracy       : {test_acc*100:.2f}%')
print(f'   Test Loss           : {test_loss:.4f}')

In [ ]:
# ── Zip & Download ────────────────────────────────────────────────
ZIP_NAME = 'submission.zip'
with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk('submission'):
        for file in files_list:
            zipf.write(os.path.join(root, file))
    zipf.write('notebook.ipynb')
    zipf.write('requirements.txt')
    zipf.write('README.md')

size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
print(f'✅ submission.zip berhasil dibuat ({size_mb:.1f} MB)')

from google.colab import files
files.download(ZIP_NAME)
print('✅ Download selesai!')